In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-987b7e31-0f58-453f-aa47-46fa6306af25;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 386ms :: artifacts dl 18ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
from pyspark.sql import functions as F
import csv
import os
# import pandas as pd
from datetime import datetime

order_items_path = "s3a://last-mile-optimization-trusted/dataset-orders/order_items_cleaned_dataset.csv/"
products_path = "s3a://last-mile-optimization-trusted/dataset-orders/products_cleaned_dataset.csv/"

order_items_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(order_items_path)
    .dropDuplicates()
    .dropna(subset=["product_id"])
    .withColumn("product_id", F.trim(F.col("product_id")))
)

products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(products_path)
    .dropDuplicates()
    .dropna(subset=["product_id"])
    .withColumn("product_id", F.trim(F.col("product_id")))
)

joined_df = order_items_df.join(products_df, on="product_id", how="inner")

joined_df.printSchema()
print("rows:", joined_df.count())

joined_df = joined_df.drop("product_id")



26/04/04 22:41:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- volume_cm3: double (nullable = true)



rows: 100516


In [4]:
joined_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/join_order_items_products.csv')

spark.stop()

26/04/04 22:42:24 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 22:42:25 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
